# run_all_0416b_nb8 — two-stage adaptive seller, switch at round 8

Part of the 0416b negotiation-data sweep (deadline-free version).

* Seller strategies covered : **8 two-stage seller schedules, switch @ round 8 of 15**
* Buyer strategies covered  : **all 10 fixed buyers**
* Mode                      : **adaptive seller × fixed buyer**
* Target per-notebook runtime : **~18 h** on `deepseek-reasoner`
* No round-limit or deadline information is included in any prompt.

The six seller strategies in the experiment universe are:
`anchor_high`, `cooperative`, `boulware_seller`, `conceder_seller`,
`tit_for_tat_matcher`, `relational`.

The ten buyer strategies are: `random`, `budget`, `wild`, `concession`,
`anchor_drag`, `boulware_buyer`, `conceder_buyer`, `tit_for_tat_buyer`,
`persistent_f0.30`, `persistent_f0.55`.


In [1]:
# ── Imports ───────────────────────────────────────────────────────────
import sys
from pathlib import Path

HERE = Path.cwd()
for p in (HERE, HERE.parent, HERE.parent.parent):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from openai import OpenAI

import exp_runner_0416 as R
from run_all_0416 import run_experiment_block

from exp_runner_0416 import (
    RandomBuyer, BudgetBuyer, WildBuyer,
    ConcessionBuyer, AnchorDragBuyer,
    BoulwareBuyer, ConcederBuyer, TitForTatBuyer,
    PersistentBuyer, AdaptiveBuyer,
    fixed_seller_schedule, twostage_seller_schedule, threestage_seller_schedule,
)
from run_all_0416 import (
    all_fixed_buyers_for_notebook, fixed_pairs_for_sellers,
)


# ── Config (edit as needed) ──────────────────────────────────────────
CLIENT = OpenAI(
    api_key="sk-81e3e6eae33d4486ad4127e0cb3630a6",          # <<< replace
    base_url="https://api.deepseek.com",
)
MODEL       = "deepseek-reasoner"
N_ROUNDS    = 15
BASE_SEED   = 5000 + 8000        # distinct seed offset per notebook
OUTPUT_ROOT = Path("results_0416b")
NOTEBOOK_ID = "nb8"


## EXPERIMENTS list — notebook 8

In [2]:
# 8 two-stage seller schedules covering hard→soft, soft→hard,
# and opposite-philosophy transitions among the 6 base strategies.
TWO_STAGE_PAIRS = [
    ("anchor_high",        "cooperative"),
    ("anchor_high",        "conceder_seller"),
    ("boulware_seller",    "conceder_seller"),
    ("boulware_seller",    "cooperative"),
    ("cooperative",        "anchor_high"),
    ("conceder_seller",    "boulware_seller"),
    ("tit_for_tat_matcher","anchor_high"),
    ("relational",         "boulware_seller"),
]

SWITCH_ROUND     = 8
N_RUNS_PER_PAIR  = 3

BUYERS = all_fixed_buyers_for_notebook()

EXPERIMENTS = []
for first, second in TWO_STAGE_PAIRS:
    sched = twostage_seller_schedule(first, second, switch_round=SWITCH_ROUND,
                                     n_rounds=N_ROUNDS)
    for b in BUYERS:
        tag = f"sel2_{first}__{second}__r{SWITCH_ROUND}__buy_{b.name}"
        EXPERIMENTS.append({
            "tag": tag, "buyer": b, "seller_schedule": sched,
            "n_runs": N_RUNS_PER_PAIR,
        })

print(f"Total conditions: {len(EXPERIMENTS)}")
print(f"Total runs      : {sum(e['n_runs'] for e in EXPERIMENTS)}")


Total conditions: 80
Total runs      : 240


## Run the block

In [3]:
# ── Execute ───────────────────────────────────────────────────────────
manifest = run_experiment_block(
    experiments=EXPERIMENTS,
    notebook_id=NOTEBOOK_ID,
    client=CLIENT,
    model=MODEL,
    n_rounds=N_ROUNDS,
    base_seed=BASE_SEED,
    output_root=OUTPUT_ROOT,
    verbose=True,
    resume=True,   # skip conditions whose JSON already exists
)
print("\n--- Per-condition timings (minutes) ---")
for t in manifest["timings"]:
    print(t)



[1/80]  sel2_anchor_high__cooperative__r8__buy_random
  buyer   = random
  seller  = anchor_high__then__cooperative@r8  (adaptive=True)
  n_runs  = 3   n_rounds = 15

  --- Run 1/3  seed=13000  prod=Cola (330ml) cost=$0.68 mkt=$2.36 int=$0.89 ---
  R 1 BUY $0.88 T5 [buy=random]  SEL $2.36 [sel=anchor_high]  "Thank you for your interest. I understand you're looking for..."
  R 2 BUY $1.14 T5 [buy=random]  SEL $2.25 [sel=anchor_high]  "Thank you for your continued interest. While I appreciate yo..."
  R 3 BUY $2.04 T3 [buy=random]  SEL $2.19 [sel=anchor_high]  "I appreciate you recognizing the value of this product by mo..."
  R 4 BUY $1.20 T5 [buy=random]  SEL $2.17 [sel=anchor_high]  "Thank you for your offer. However, $1.20 per unit is not a w..."
  R 5 BUY $1.08 T6 [buy=random]  SEL $2.15 [sel=anchor_high]  "I appreciate your persistence, but $1.08 per unit is not a f..."
  R 6 BUY $2.14 T5 [buy=random]  SEL $2.14 [sel=anchor_high]  "I appreciate you aligning your offer much closer 

## Inspection

In [4]:
# ── Quick inspection of the first run of the first condition ────────
import json
first_tag = EXPERIMENTS[0]["tag"]
with open(OUTPUT_ROOT / NOTEBOOK_ID / f"{first_tag}.json") as f:
    exp = json.load(f)
R.inspect_run(exp["runs"][0], max_chars=200)


UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 3907: illegal multibyte sequence